<a href="https://colab.research.google.com/github/anashenkonina-collab/python_for_ds_task/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22HW_15_4_%D0%90%D0%BD%D0%B0%D0%BB%D1%96%D0%B7_%D0%90_%D0%92_%D1%82%D0%B5%D1%81%D1%82%D1%96%D0%B2_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [6]:
import math
import statsmodels.stats.api as sms

# Вихідні параметри утримання (retention)
p1 = 0.19  # базове утримання (19%)
p2 = 0.20  # очікуване утримання після змін (19% + 1%)

# Розмір ефекту (Effect Size) для двох пропорцій
effect_size = sms.proportion_effectsize(p1, p2)

# Необхідний розмір вибірки для ОДНІЄЇ групи
n_per_group = sms.NormalIndPower().solve_power(
    effect_size=effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
)

# Округлення в більшу сторону до цілого користувача
n_per_group = math.ceil(n_per_group)
total_users = n_per_group * 2

print(f"Користувачів на одну групу: {n_per_group}")
print(f"Сумарно користувачів для обох груп (gate_30 + gate_40): {total_users}")


Користувачів на одну групу: 24638
Сумарно користувачів для обох груп (gate_30 + gate_40): 49276


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [7]:
import pandas as pd

df = pd.read_csv('cookie_cats.csv')

# Середнє значення retention_7 за версіями гри
retention_7_by_version = df.groupby('version')['retention_7'].mean()

print("Середнє утримання на 7-й день (частка):")
print(retention_7_by_version)

print("\nУ відсотках:")
print((retention_7_by_version * 100).round(2))


Середнє утримання на 7-й день (частка):
version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64

У відсотках:
version
gate_30    19.02
gate_40    18.20
Name: retention_7, dtype: float64


### Аналіз та формулювання гіпотези (Завдання 2)

За попередніми середніми значеннями у вибірці версія **gate_30** (19.02%) показує краще утримання гравців на 7-й день, ніж версія **gate_40** (18.20%).

На основі цього формулюємо статистичні гіпотези для перевірки:
* **Нульова гіпотеза ($H_0$):** переміщення воріт не впливає на утримання гравців на 7-й день, показники обох версій однакові ($p_{30} = p_{40}$).
* **Альтернативна гіпотеза ($H_a$):** версія **gate_30** дає вище утримання на 7-й день порівняно з **gate_40** ($p_{30} > p_{40}$), або ж утримання між версіями статистично значущо відрізняється ($p_{30} \neq p_{40}$).


3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [8]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

# Розділення даних на контрольну (gate_30) та тестову (gate_40) групи
control = df[df['version'] == 'gate_30']['retention_7']
treatment = df[df['version'] == 'gate_40']['retention_7']

# Кількість успіхів (повернень на 7-й день) та загальну кількість гравців
successes = [control.sum(), treatment.sum()]
nobs = [control.count(), treatment.count()]

# Двосторонній Z-тест для двох пропорцій
z_stat, p_value = proportions_ztest(successes, nobs, alternative='two-sided')

# 95% довірчі інтервали (alpha = 0.05) для кожної групи
ci_control = proportion_confint(control.sum(), control.count(), alpha=0.05, method='normal')
ci_treatment = proportion_confint(treatment.sum(), treatment.count(), alpha=0.05, method='normal')

print(f"z statistic: {z_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Довірчий інтервал 95% для групи control: [{ci_control[0]:.4f}, {ci_control[1]:.4f}]")
print(f"Довірчий інтервал 95% для групи treatment: [{ci_treatment[0]:.4f}, {ci_treatment[1]:.4f}]")


z statistic: 3.1644
p-value: 0.0016
Довірчий інтервал 95% для групи control: [0.1866, 0.1938]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1855]


### Висновки до Завдання 3

1. **Чи є статистична значуща різниця між поведінкою користувачів у різних версіях гри?**
   * **Так, різниця є статистично значущою.** Отримане значення $p\text{-value} = 0.0016$ є значно меншим за встановлений рівень значущості $\alpha = 0.05$ (а $|Z| = 3.1644 > 1.96$). Тому ми відхиляємо нульову гіпотезу. Версія **gate_30** (контрольна група) статистично значущо краще утримує гравців на 7-й день, ніж версія **gate_40**.

2. **Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?**
   * **Ні, довірчі інтервали не перетинаються:** для групи `control` (`gate_30`) інтервал становить **[18.66%, 19.38%]**, а для групи `treatment` (`gate_40`) — **[17.85%, 18.55%]**. Навіть нижня межа контрольної групи (0.1866) вища за верхню межу тестової групи (0.1855).
   * **Що це означає:** Відсутність перетину візуально та математично підтверджує наявність статистично значущої різниці між групами. Переміщення воріт на 40-й рівень справді погіршує утримання гравців, тому зміну впроваджувати не варто.


4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


In [9]:
import pandas as pd
from scipy.stats import chi2_contingency

# Таблиця спряженості (кількість гравців по групах та результатах утримання)
contingency_table = pd.crosstab(df['version'], df['retention_7'])
print("Таблиця спряженості:")
print(contingency_table)

# Тест Хі-квадрат
chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table, correction=False)

print("РЕЗУЛЬТАТИ ТЕСТУ ХІ-КВАДРАТ")
print(f"Статистика Хі-квадрат: {chi2_stat:.4f}")
print(f"p-value: {p_value:.4f}")


Таблиця спряженості:
retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279
РЕЗУЛЬТАТИ ТЕСТУ ХІ-КВАДРАТ
Статистика Хі-квадрат: 10.0132
p-value: 0.0016


### Аналіз та висновки до Завдання 4 (Тест Хі-квадрат)

1. **Формулювання гіпотез:**
   * **Нульова гіпотеза ($H_0$):** утримання гравця на 7-й день є незалежним від версії гри (переміщення воріт не впливає на поведінку гравців).
   * **Альтернативна гіпотеза ($H_a$):** існує статистично значуща залежність між версією гри та утриманням гравця на 7-й день.

2. **Результати та висновок ($\alpha = 0.05$):**
   * Отримане значення $p\text{-value} = 0.0016$ є значно меншим за встановлений рівень значущості $\alpha = 0.05$ (статистика $\chi^2 \approx 10.01$).
   * **Рішення:** Ми **відхиляємо нульову гіпотезу ($H_0$)**. Це підтверджує наявність статистично значущої залежності між розміщенням воріт та утриманням гравців на 7-й день. Результат тесту Хі-квадрат повністю узгоджується з результатами попереднього Z-тесту.
